In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("paper", font_scale=1.2)
sns.set_style("whitegrid")

In [ ]:
# Load data
indir = "/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/metrics_new_donor_cell_eval"

df_per_pert = pl.read_csv(
    os.path.join(indir, "cell_eval_metrics_per_perturbation.csv"),
    infer_schema_length=10000,
    schema_overrides={"split_or_wandb": pl.Utf8},
).to_pandas()

print(f"Per-perturbation shape: {df_per_pert.shape}")
print(f"Methods: {sorted(df_per_pert['method'].unique())}")
print(f"Donors: {sorted(df_per_pert['donor'].unique())}")
print(f"Counts per method:")
print(df_per_pert.groupby('method').size())

In [ ]:
# Map method names to display names
method_display = {
    "cellflow": "CellFlow",
    "mean_model_1": "Mean model 1",
    "mean_model_2": "Mean model 2",
    "identity": "Identity",
    "closest_embedding": "Closest embedding",
}

color_dict = {
    "CellFlow": "#B12F8C",
    "Mean model 1": "#8F97A8",
    "Mean model 2": "#566573",
    "Identity": "#BDBDBD",
    "Closest embedding": "#E0E0E0",
}

method_order = ["CellFlow", "Closest embedding", "Mean model 1", "Mean model 2", "Identity"]

df_per_pert["model"] = df_per_pert["method"].map(method_display)

# For methods with multiple seeds/splits, average across them per (donor, perturbation)
meta_cols = ["model", "donor", "perturbation"]
metric_cols = [
    "pearson_delta", "mse_delta", "mae_delta",
    "discrimination_score_l1", "discrimination_score_l2", "discrimination_score_cosine",
    "pearson_edistance", "clustering_agreement",
    "overlap_at_N", "precision_at_N",
    "de_spearman_sig", "de_direction_match", "de_spearman_lfc_sig",
    "de_sig_genes_recall", "pr_auc", "roc_auc",
]

df = df_per_pert.groupby(meta_cols)[metric_cols].mean().reset_index()
print(f"Averaged shape: {df.shape}")
print(f"Counts per model:")
print(df.groupby('model').size())

# Distribution metrics

In [ ]:
dist_metrics = [
    ("pearson_delta", "Pearson (delta)"),
    ("mse_delta", "MSE (delta)"),
    ("mae_delta", "MAE (delta)"),
    ("pearson_edistance", "Pearson E-distance"),
    ("clustering_agreement", "Clustering agreement"),
]

fig, axes = plt.subplots(1, len(dist_metrics), figsize=(4 * len(dist_metrics), 4))
for ax, (col, title) in zip(axes, dist_metrics):
    data = df.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title}\n(no data)")
        continue
    available = [m for m in method_order if m in data["model"].unique()]
    sns.boxplot(
        data=data, x="model", y=col, order=available,
        palette=color_dict, ax=ax, showfliers=False,
    )
    sns.stripplot(
        data=data, x="model", y=col, order=available,
        color=".3", size=2, alpha=0.5, ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
outdir = "/ictstr01/home/icb/dominik.klein/git_repos/ot_pert_new/fig_2/revision/cell_eval"
fig.savefig(os.path.join(outdir, "cell_eval_distribution_metrics.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(outdir, "cell_eval_distribution_metrics.pdf"), bbox_inches="tight")
plt.show()

# Discrimination metrics

In [ ]:
disc_metrics = [
    ("discrimination_score_l1", "Discrimination (L1)"),
    ("discrimination_score_l2", "Discrimination (L2)"),
    ("discrimination_score_cosine", "Discrimination (cosine)"),
]

fig, axes = plt.subplots(1, len(disc_metrics), figsize=(4 * len(disc_metrics), 4))
for ax, (col, title) in zip(axes, disc_metrics):
    data = df.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title}\n(no data)")
        continue
    available = [m for m in method_order if m in data["model"].unique()]
    sns.boxplot(
        data=data, x="model", y=col, order=available,
        palette=color_dict, ax=ax, showfliers=False,
    )
    sns.stripplot(
        data=data, x="model", y=col, order=available,
        color=".3", size=2, alpha=0.5, ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(os.path.join(outdir, "cell_eval_discrimination_metrics.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(outdir, "cell_eval_discrimination_metrics.pdf"), bbox_inches="tight")
plt.show()

# DE metrics

In [ ]:
de_metrics = [
    ("overlap_at_N", "Overlap at N"),
    ("precision_at_N", "Precision at N"),
    ("de_spearman_sig", "DE Spearman (sig)"),
    ("de_direction_match", "DE direction match"),
    ("de_sig_genes_recall", "DE sig genes recall"),
    ("pr_auc", "PR AUC"),
    ("roc_auc", "ROC AUC"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, (col, title) in enumerate(de_metrics):
    ax = axes[i]
    data = df.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title}\n(no data)")
        continue
    available = [m for m in method_order if m in data["model"].unique()]
    sns.boxplot(
        data=data, x="model", y=col, order=available,
        palette=color_dict, ax=ax, showfliers=False,
    )
    sns.stripplot(
        data=data, x="model", y=col, order=available,
        color=".3", size=2, alpha=0.5, ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
# Hide unused subplot
axes[-1].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(outdir, "cell_eval_de_metrics.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(outdir, "cell_eval_de_metrics.pdf"), bbox_inches="tight")
plt.show()

# Summary bar plot (mean across perturbations)

In [ ]:
summary_metrics = [
    ("pearson_delta", "Pearson (delta)", True),
    ("mse_delta", "MSE (delta)", False),
    ("discrimination_score_l2", "Discrimination (L2)", False),
    ("clustering_agreement", "Clustering agreement", True),
    ("de_direction_match", "DE direction match", True),
    ("pr_auc", "PR AUC", True),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, (col, title, higher_better) in enumerate(summary_metrics):
    ax = axes[i]
    data = df.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title}\n(no data)")
        continue
    available = [m for m in method_order if m in data["model"].unique()]
    sns.barplot(
        data=data, x="model", y=col, order=available,
        palette=color_dict, ax=ax, errorbar="se",
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(os.path.join(outdir, "cell_eval_summary.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(outdir, "cell_eval_summary.pdf"), bbox_inches="tight")
plt.show()

# Summary table (mean +/- std per model)

In [ ]:
summary_cols = [
    "pearson_delta", "mse_delta", "clustering_agreement",
    "discrimination_score_l2", "de_direction_match", "pr_auc", "roc_auc",
]

summary = df.groupby("model")[summary_cols].agg(["mean", "std"]).round(4)
summary.columns = [f"{col}_{stat}" for col, stat in summary.columns]
summary = summary.loc[[m for m in method_order if m in summary.index]]
print(summary.to_string())